In [ ]:
import pandas as pd
import os
import subprocess
import shutil
import glob
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests

# === CONFIGURATION ===
MAX_PROJECTS = 5
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# === LOAD .env ===
load_dotenv(ENV_FILE)
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

START_NUMBER = int(os.getenv("START_NUMBER", "1"))
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# === PATHS ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
base_dir = Path(r"E:\Android Mobile Project\AndroidProjects")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
commits_dir = base_dir / "Commits"
build_info_dir = base_dir / "BuildInfo"
metadata_csv = base_dir / "8.2-Project_Metadata.csv"

# === CLEAN OLD DATA ===
for path in [clone_dir, cloned_sample_dir, yml_output_dir, commits_dir, build_info_dir]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")].reset_index(drop=True)

# === HANDLE SAMPLE_LIST ===
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# === PREP METADATA COLLECTION ===
metadata_rows = []

# === PROCESS EACH REPO ===
for i in range(START_NUMBER - 1, len(df)):
    url = df.loc[i, 'github_url']
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_name = f"{username}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name}...")

    try:
        subprocess.run(['git', 'clone', '--depth', '1', url, str(repo_path)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except subprocess.TimeoutExpired:
        print(f"⏱️ Timeout while cloning {repo_name}, skipping...")
        continue
    print("✅ Clone complete")

    # --- Extract .yml/.yaml Files ---
    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(('.yml', '.yaml')):
                full_path = Path(root) / file
                rel_path = full_path.relative_to(repo_path)
                safe_name = f"{repo_name}.{str(rel_path).replace(os.sep, '_')}"
                shutil.copy2(full_path, yml_output_dir / safe_name)
    print("📄 Config files extracted")

    # --- Extract build.gradle test lines ---
    build_info_dir.mkdir(parents=True, exist_ok=True)
    info_path = build_info_dir / f"{repo_name}_build_info.txt"
    with open(info_path, 'w', encoding='utf-8') as out_file:
        for gradle_file in glob.glob(str(repo_path / '**/*.gradle*'), recursive=True):
            try:
                with open(gradle_file, 'r', encoding='utf-8', errors='ignore') as f:
                    lines = f.readlines()
                    test_lines = [line for line in lines if 'test' in line.lower()]
                    if test_lines:
                        out_file.write(f"\n--- {gradle_file} ---\n")
                        out_file.writelines(test_lines)
            except Exception:
                continue
    print("🛠️ Build info extracted")

    # --- Save Commit Log ---
    with open(commits_dir / f"{repo_name}_commits.txt", 'w', encoding='utf-8') as f:
        subprocess.run(['git', 'log', '--pretty=format:%h | %an | %ad | %s'], cwd=repo_path, stdout=f, stderr=subprocess.DEVNULL)
    print("📜 Commits saved")

    # --- Save Contributors via GitHub API ---
    try:
        api_url = f"https://api.github.com/repos/{username}/{project}/contributors"
        r = requests.get(api_url, headers={'Authorization': f'token {GITHUB_TOKEN}'}, timeout=30)
        if r.ok:
            contributors_data = r.json()
            contrib_path = commits_dir / f"{repo_name}_contributors.txt"
            with open(contrib_path, 'w', encoding='utf-8') as f:
                for contributor in contributors_data:
                    f.write(f"{contributor['contributions']:>4} | {contributor['login']}\n")
            print("👥 Contributors saved via GitHub API")
        else:
            print(f"⚠️ GitHub API error ({r.status_code}) for contributors of {repo_name}")
    except Exception as e:
        print(f"⚠️ Error retrieving contributors for {repo_name}: {e}")

    # --- GitHub API Metadata (Full Metrics) ---
    try:
        r = requests.get(f"https://api.github.com/repos/{username}/{project}", headers={'Authorization': f'token {GITHUB_TOKEN}'}, timeout=30)
        if r.ok:
            data = r.json()

            # Pull request count
            pr_count = 0
            pr_url = f"https://api.github.com/repos/{username}/{project}/pulls?state=all&per_page=1"
            pr_resp = requests.get(pr_url, headers={'Authorization': f'token {GITHUB_TOKEN}'})
            if 'Link' in pr_resp.headers:
                pr_count = int(pr_resp.headers['Link'].split('page=')[-1].split('>')[0])
            else:
                pr_count = len(pr_resp.json())

            # Commit count
            commit_count = 0
            commit_url = f"https://api.github.com/repos/{username}/{project}/commits?per_page=1"
            commit_resp = requests.get(commit_url, headers={'Authorization': f'token {GITHUB_TOKEN}'})
            if 'Link' in commit_resp.headers:
                commit_count = int(commit_resp.headers['Link'].split('page=')[-1].split('>')[0])
            else:
                commit_count = len(commit_resp.json())

            # Contributors count
            contrib_url = f"https://api.github.com/repos/{username}/{project}/contributors"
            contrib_resp = requests.get(contrib_url, headers={'Authorization': f'token {GITHUB_TOKEN}'})
            contrib_count = len(contrib_resp.json()) if contrib_resp.ok else 0

            metadata_rows.append({
                'repo_name': repo_name,
                'full_name': data.get('full_name'),
                'description': data.get('description'),
                'language': data.get('language'),
                'license': data.get('license', {}).get('name') if data.get('license') else None,
                'created_at': data.get('created_at'),
                'updated_at': data.get('updated_at'),
                'last_commit_date': data.get('pushed_at'),
                'stars': data.get('stargazers_count'),
                'forks': data.get('forks_count'),
                'watchers': data.get('watchers_count'),
                'open_issues': data.get('open_issues_count'),
                'contributors': contrib_count,
                'pull_requests': pr_count,
                'commits': commit_count,
                'size': data.get('size')
            })
            print("📊 Full metrics retrieved")
    except Exception as e:
        print(f"⚠️ Exception retrieving metadata for {repo_name}: {e}")

    # --- Move Sample or Delete Repo ---
    try:
        if i in sample_indices_to_keep:
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, ignore_errors=True)
            shutil.move(str(repo_path), str(dest_path))
            print(f"📦 Sample repo moved to: {dest_path}")
        else:
            shutil.rmtree(repo_path, ignore_errors=True)
            print(f"🗑️ Non-sample repo deleted: {repo_path.name}")
    except Exception as e:
        print(f"❌ Error handling repo folder for {repo_name}: {e}")

    # --- Save updated START_NUMBER ---
    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))

# === SAVE METADATA CSV ===
if metadata_rows:
    pd.DataFrame(metadata_rows).to_csv(metadata_csv, index=False)
    print(f"\n✅ Saved metadata for {len(metadata_rows)} projects")

print("\n🏁 Finished processing selected projects.")
